# DataFrameSplitter

This Encoder splits a DataFrame into multiple DataFrames / Series


In [1]:
%config InteractiveShell.ast_node_interactivity='last_expr_or_assign'  # always print last expr.
%config InlineBackend.figure_format = 'svg'
%load_ext autoreload
%autoreload 2
%matplotlib inline

# import logging
# logging.basicConfig(level=logging.INFO)

In [2]:
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
from pandas import *
np.set_printoptions(precision=4, floatmode='fixed', suppress=True)
rng = np.random.default_rng()

Generator(PCG64) at 0x7FD661EDDBA0

In [3]:
from tsdm.encoders.modular import *
from tsdm.tasks import KIWI_FINAL_PRODUCT
task = KIWI_FINAL_PRODUCT()
ts = task.timeseries.sort_index(axis="index").sort_index(axis="columns")

variable                                  Acetate  Base  \
run_id experiment_id measurement_time                     
439    15325         2020-12-09 09:10:09      NaN   NaN   
                     2020-12-09 09:10:19      NaN   NaN   
                     2020-12-09 09:10:24      NaN   NaN   
                     2020-12-09 09:10:25      NaN   NaN   
                     2020-12-09 09:10:34      NaN   NaN   
...                                           ...   ...   
510    16871         2021-10-26 22:41:26      NaN   NaN   
                     2021-10-26 22:42:11      NaN   NaN   
                     2021-10-26 22:42:20      NaN   NaN   
                     2021-10-26 22:43:22      NaN   NaN   
                     2021-10-26 22:43:31      NaN   NaN   

variable                                  Cumulated_feed_volume_glucose  \
run_id experiment_id measurement_time                                     
439    15325         2020-12-09 09:10:09                            0.0   
                     2020-12-09 09:10:19                            0.0   
                     2020-12-09 09:10:24                            0.0   
                     2020-12-09 09:10:25                            0.0   
                     2020-12-09 09:10:34                            0.0   
...                                                                 ...   
510    16871         2021-10-26 22:41:26                         1061.0   
                     2021-10-26 22:42:11                         1061.0   
                     2021-10-26 22:42:20                         1061.0   
                     2021-10-26 22:43:22                         1061.0   
                     2021-10-26 22:43:31                         1061.0   

variable                                  Cumulated_feed_volume_medium  \
run_id experiment_id measurement_time                                    
439    15325         2020-12-09 09:10:09                      0.000000   
                     2020-12-09 09:10:19                      0.000000   
                     2020-12-09 09:10:24                      0.000000   
                     2020-12-09 09:10:25                      0.000000   
                     2020-12-09 09:10:34                      0.000000   
...                                                                ...   
510    16871         2021-10-26 22:41:26                   3787.307373   
                     2021-10-26 22:42:11                   3787.307373   
                     2021-10-26 22:42:20                   3787.307373   
                     2021-10-26 22:43:22                   3787.307373   
                     2021-10-26 22:43:31                   3787.307373   

variable                                        DOT  Flow_Air  Fluo_GFP  \
run_id experiment_id measurement_time                                     
439    15325         2020-12-09 09:10:09        NaN       0.0       NaN   
                     2020-12-09 09:10:19   0.000000       0.0       NaN   
                     2020-12-09 09:10:24        NaN       0.0       NaN   
                     2020-12-09 09:10:25        NaN       0.0       NaN   
                     2020-12-09 09:10:34   0.000000       0.0       NaN   
...                                             ...       ...       ...   
510    16871         2021-10-26 22:41:26        NaN       5.0       NaN   
                     2021-10-26 22:42:11        NaN       5.0       NaN   
                     2021-10-26 22:42:20  79.459999       5.0       NaN   
                     2021-10-26 22:43:22        NaN       5.0       NaN   
                     2021-10-26 22:43:31  78.860001       5.0       NaN   

variable                                  Glucose  InducerConcentration  \
run_id experiment_id measurement_time                                     
439    15325         2020-12-09 09:10:09      NaN                   0.0   
                     2020-12-09 09:10:19      NaN                   0.0   
                 

In [4]:
channel_freq = pd.notna(ts).mean().sort_values()

fast_channels = channel_freq[channel_freq >= .1].index
slow_channels = channel_freq[channel_freq <  .1].index
FAST = ts[fast_channels].dropna(how="all")
SLOW = ts[slow_channels].dropna(how="all")
groups = {"fast": fast_channels, "slow": slow_channels}

{'fast': Index(['Temperature', 'DOT', 'pH', 'Cumulated_feed_volume_glucose',
        'Cumulated_feed_volume_medium', 'Flow_Air', 'InducerConcentration',
        'Probe_Volume', 'StirringSpeed'],
       dtype='object', name='variable'),
 'slow': Index(['OD600', 'Fluo_GFP', 'Glucose', 'Acetate', 'Base', 'Volume'], dtype='object', name='variable')}

In [9]:
from collections.abc import Hashable, Iterable, Sequence
from pandas import Series, DataFrame
from typing import Union, Literal, Any
from collections import namedtuple

In [11]:
class DataFrameSplitter(BaseEncoder):
    columns: Index
    dtypes: Series
    groups: dict[Any, Sequence[Any]]

    @staticmethod
    def _pairwise_disjoint(groups: Iterable[Sequence[Any]]) -> bool:
        union: set[HashableType] = set().union(*(set(obj) for obj in groups))
        n = sum(len(u) for u in groups)
        return n == len(union)

    def __init__(self, groups: dict[Any, Sequence[Any]]) -> None:
        super().__init__()
        self.groups = groups
        assert self._pairwise_disjoint(self.groups.values())

    def fit(self, data) -> None:
        self.columns = data.columns
        self.dtypes = data.dtypes

    def encode(self, data: DataFrame) -> tuple[DataFrame, ...]:
        encoded = []
        for group, columns in self.groups.items():
            encoded.append(data[columns].dropna(how="all"))
        return tuple(encoded)

    def decode(self, data: tuple[DataFrame, ...]) -> DataFrame:
        decoded = pd.concat(data, axis="columns")
        decoded = decoded.astype(self.dtypes)
        decoded = decoded[self.columns]
        return decoded

In [12]:
encoder = DataFrameSplitter(groups)
encoder.fit(ts)
encoded = encoder.encode(ts)
decoded = encoder.decode(encoded)
pd.testing.assert_frame_equal(ts, decoded)

In [15]:
encoded

(                                                value  Temperature  DOT  pH  \
 run_id experiment_id measurement_time                                         
 439    15325         2020-12-09 09:10:09    36.389999            1    0   0   
                      2020-12-09 09:10:09     0.000000            0    0   0   
                      2020-12-09 09:10:09     0.000000            0    0   0   
                      2020-12-09 09:10:09     0.000000            0    0   0   
                      2020-12-09 09:10:09     0.000000            0    0   0   
 ...                                               ...          ...  ...  ..   
 510    16871         2021-10-26 22:43:31  3787.307373            0    0   0   
                      2021-10-26 22:43:31     5.000000            0    0   0   
                      2021-10-26 22:43:31     0.000000            0    0   0   
                      2021-10-26 22:43:31  5500.000000            0    0   0   
                      2021-10-26 22:43:3

In [13]:
enc = TripletEncoder()
enc.fit(encoded[0])
enc.encode(encoded[0])

value  Temperature  DOT  pH  \
run_id experiment_id measurement_time                                         
439    15325         2020-12-09 09:10:09    36.389999            1    0   0   
                     2020-12-09 09:10:09     0.000000            0    0   0   
                     2020-12-09 09:10:09     0.000000            0    0   0   
                     2020-12-09 09:10:09     0.000000            0    0   0   
                     2020-12-09 09:10:09     0.000000            0    0   0   
...                                               ...          ...  ...  ..   
510    16871         2021-10-26 22:43:31  3787.307373            0    0   0   
                     2021-10-26 22:43:31     5.000000            0    0   0   
                     2021-10-26 22:43:31     0.000000            0    0   0   
                     2021-10-26 22:43:31  5500.000000            0    0   0   
                     2021-10-26 22:43:31     0.000000            0    0   0   

                                          Cumulated_feed_volume_glucose  \
run_id experiment_id measurement_time                                     
439    15325         2020-12-09 09:10:09                              0   
                     2020-12-09 09:10:09                              1   
                     2020-12-09 09:10:09                              0   
                     2020-12-09 09:10:09                              0   
                     2020-12-09 09:10:09                              0   
...                                                                 ...   
510    16871         2021-10-26 22:43:31                              0   
                     2021-10-26 22:43:31                              0   
                     2021-10-26 22:43:31                              0   
                     2021-10-26 22:43:31                              0   
                     2021-10-26 22:43:31                              0   

                                          Cumulated_feed_volume_medium  \
run_id experiment_id measurement_time                                    
439    15325         2020-12-09 09:10:09                             0   
                     2020-12-09 09:10:09                             0   
                     2020-12-09 09:10:09                             1   
                     2020-12-09 09:10:09                             0   
                     2020-12-09 09:10:09                             0   
...                                                                ...   
510    16871         2021-10-26 22:43:31                             1   
                     2021-10-26 22:43:31                             0   
                     2021-10-26 22:43:31                             0   
                     2021-10-26 22:43:31                             0   
                     2021-10-26 22:43:31                             0   

                                          Flow_Air  InducerConcentration  \
run_id experiment_id measurement_time                                      
439    15325         2020-12-09 09:10:09         0                     0   
                     2020-12-09 09:10:09         0                     0   
                     2020-12-09 09:10:09         0                     0   
                     2020-12-09 09:10:09         1                     0   
                     2020-12-09 09:10:09         0                     1   
...                                            ...                   ...   
510    16871         2021-10-26 22:43:31         0                     0   
                     2021-10-26 22:43:31         1                     0   
                     2021-10-26 22:43:31         0                     1   
                     2021-10-26 22:43:31         0                     0   
                     2021-10-26 22:43:31         0                     0   

                                          Probe_Volume  StirringSpeed  
run_id experiment_id

In [14]:
encoder = (TripletEncoder() | TripletEncoder()) @ DataFrameSplitter(groups)
encoder.fit(ts)
encoded = encoder.encode(ts)
decoded = encoder.decode(encoded)
pd.testing.assert_frame_equal(decoded, ts)